<a href="https://colab.research.google.com/github/hanokjoshua144/DEEP-LEARNING-6-SEM/blob/main/W_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implement RNN for predicting the next character,word and sentence.


RNN – Next Character Prediction

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Sample text
text = "hello world"
chars = list(set(text))
char2idx = {ch:i for i,ch in enumerate(chars)}
idx2char = {i:ch for ch,i in char2idx.items()}

# Prepare dataset
seq_length = 3
X = []
y = []

for i in range(len(text) - seq_length):
    seq = text[i:i+seq_length]
    target = text[i+seq_length]
    X.append([char2idx[c] for c in seq])
    y.append(char2idx[target])

X = torch.tensor(X)
y = torch.tensor(y)

# Model
class CharRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.rnn = nn.RNN(vocab_size, 16, batch_first=True)
        self.fc = nn.Linear(16, vocab_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return out

model = CharRNN(len(chars))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# One-hot encoding
def one_hot(x, vocab_size):
    return torch.eye(vocab_size)[x]

# Training
for epoch in range(200):
    inputs = one_hot(X, len(chars)).float()
    outputs = model(inputs)

    loss = criterion(outputs, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Prediction
def predict(seq):
    x = torch.tensor([[char2idx[c] for c in seq]])
    x = one_hot(x, len(chars)).float()
    out = model(x)
    return idx2char[torch.argmax(out).item()]

print(predict("hel"))

2. RNN – Next Word Prediction

In [ ]:
import torch
import torch.nn as nn

text = "i love deep learning and i love ai"

words = text.split()
vocab = list(set(words))
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

seq_length = 2
X, y = [], []

for i in range(len(words) - seq_length):
    X.append([word2idx[w] for w in words[i:i+seq_length]])
    y.append(word2idx[words[i+seq_length]])

X = torch.tensor(X)
y = torch.tensor(y)

class WordRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 10)
        self.rnn = nn.RNN(10, 16, batch_first=True)
        self.fc = nn.Linear(16, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return out

model = WordRNN(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    output = model(X)
    loss = criterion(output, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

def predict(seq):
    x = torch.tensor([[word2idx[w] for w in seq]])
    out = model(x)
    return idx2word[torch.argmax(out).item()]

print(predict(["i","love"]))

3. RNN – Sentence Prediction (Sequence Generation)

In [ ]:
import torch
import torch.nn as nn

text = "deep learning is fun and deep learning is powerful"
words = text.split()

vocab = list(set(words))
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

class SentenceRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 16)
        self.rnn = nn.RNN(16, 32, batch_first=True)
        self.fc = nn.Linear(32, vocab_size)

    def forward(self, x, hidden):
        x = self.embed(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

model = SentenceRNN(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training
for epoch in range(200):
    loss = 0
    hidden = None

    for i in range(len(words)-1):
        x = torch.tensor([[word2idx[words[i]]]])
        y = torch.tensor([word2idx[words[i+1]]])

        output, hidden = model(x, hidden)
        loss += criterion(output.view(-1, len(vocab)), y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Generate sentence
def generate(start_word, length=5):
    word = start_word
    result = [word]
    hidden = None

    for _ in range(length):
        x = torch.tensor([[word2idx[word]]])
        out, hidden = model(x, hidden)
        word = idx2word[torch.argmax(out).item()]
        result.append(word)

    return " ".join(result)

print(generate("deep"))

4. LSTM Implementation

In [ ]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 16)
        self.lstm = nn.LSTM(16, 32, batch_first=True)
        self.fc = nn.Linear(32, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

5. GRU Implementation

In [ ]:
import torch
import torch.nn as nn

class GRUModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 16)
        self.gru = nn.GRU(16, 32, batch_first=True)
        self.fc = nn.Linear(32, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.gru(x)
        out = self.fc(out[:, -1, :])
        return out

6. Encoder–Decoder Model (Translation)

In [ ]:
import torch
import torch.nn as nn

# Toy dataset
input_vocab = {"i":0, "love":1, "you":2}
target_vocab = {"je":0, "t'aime":1}

input_seq = torch.tensor([[0,1,2]])   # i love you
target_seq = torch.tensor([[0,1]])    # je t'aime

# Encoder
class Encoder(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.embed = nn.Embedding(input_size, 16)
        self.rnn = nn.LSTM(16, 32, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        outputs, (hidden, cell) = self.rnn(x)
        return hidden, cell

# Decoder
class Decoder(nn.Module):
    def __init__(self, output_size):
        super().__init__()
        self.embed = nn.Embedding(output_size, 16)
        self.rnn = nn.LSTM(16, 32, batch_first=True)
        self.fc = nn.Linear(32, output_size)

    def forward(self, x, hidden, cell):
        x = self.embed(x)
        outputs, (hidden, cell) = self.rnn(x, (hidden, cell))
        predictions = self.fc(outputs)
        return predictions, hidden, cell

encoder = Encoder(len(input_vocab))
decoder = Decoder(len(target_vocab))

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)

# Training
for epoch in range(200):
    hidden, cell = encoder(input_seq)

    output, _, _ = decoder(target_seq, hidden, cell)

    loss = criterion(output.view(-1, len(target_vocab)), target_seq.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Loss: {loss.item()}")

OBSERVATIONS


RNN – Next Character Prediction (Observations)
The model learns character-level patterns from the input text.
It successfully predicts the next character based on previous sequence.
Captures short-term dependencies (like “hel → l”).
Performance depends on sequence length and training data size.
Struggles with long-range dependencies due to vanishing gradients.
Output is often probabilistic, not always perfectly accurate.


 RNN – Next Word Prediction (Observations)
Learns word-level relationships in sentences.
Predicts the next word based on context of previous words.
Embedding layer helps convert words into dense vector representations.
Works well for simple and repetitive datasets.
Fails to capture long-term context in longer sentences.
Vocabulary size impacts model complexity and accuracy.



 RNN – Sentence Generation (Observations)
Model generates sequence of words step-by-step.
Output depends heavily on starting word (seed input).
Can produce coherent short sentences.
May generate repetitive or meaningless sequences over longer outputs.
Hidden state carries information from previous time steps.
Suffers from error accumulation during long generation.



LSTM (Observations)
Handles long-term dependencies better than RNN.
Uses gates: input, forget, output gates to control information flow.
Reduces vanishing gradient problem.
Maintains a cell state for long-term memory.
Produces more stable and accurate predictions than RNN.
Computationally heavier than basic RNN.



 GRU (Observations)
Similar to LSTM but with simpler architecture.
Uses only update and reset gates.
Requires fewer parameters → faster training.
Performs almost as well as LSTM in many tasks.
Easier to implement and less computational cost.
Suitable when speed is important.


 Encoder–Decoder Model (Observations)
Converts input sequence into a fixed-length context vector.
Decoder generates output sequence step-by-step.
Useful for machine translation and sequence-to-sequence tasks.
Performance depends on quality of encoded representation.
Struggles with long sentences (information bottleneck).
Can be improved using attention mechanisms.